# 30-Day Hospital Readmission Prediction : Diabetic Patients

This notebook predicts 30-day readmission in the UCI diabetic cohort using leakage-aware preprocessing, grouped nested validation, imbalance handling, calibration, explainability, and Bayesian benchmarking. Huang et al. (2021), Le et al. (2025), and Hasan et al. (2010) motivate this design and reporting scope.

References
Huang et al. (2021). Application of machine learning in predicting hospital readmission. *BMC Med Res Methodol*. 2021;21:96.
Le et al. (2025). Machine learning-based prediction of unplanned readmission. *Cancer Control*. 2025;32:10732748251332803.
Hasan et al. (2010). Hospital readmission in general medicine patients. *J Gen Intern Med*. 2010;25(3):211219.

In [ ]:
# Core imports and project path setup

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import json

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns

import statsmodels.api as sm

from sklearn.model_selection import StratifiedGroupKFold

from sklearn.metrics import brier_score_loss

from readmission_analysis_ieee import (

    RANDOM_STATE,

    ACTIVE_THREADS,

    OUTER_FOLDS,

    CALIBRATION_TOP_MODELS,

    BAYES_FEATURE_LIMIT,

    BAYES_SUBSET_SIZE,

    REPEATED_CV_REPEATS,

    load_ids_mapping,

    save_json,

    EncodedDataBundle,

    CombinedFeatureSelector,

    maybe_apply_smote,

    prepare_fold_datasets,

    nested_cv_for_model,

    aggregate_best_params,

    build_model,

    compute_metrics,

    expected_calibration_error,

    fit_calibration_models,

    shap_analysis,

    fit_bayesian_logistic,

    fit_bayesian_bart,

    decision_curve_frame,

    plot_numeric_distributions,

    plot_correlation_heatmap,

    plot_category_frequencies,

    plot_roc_curves,

    plot_pr_curves,

    plot_confusion_matrices,

    plot_calibration_curves,

    plot_decision_curves,

    write_results_report,

)

np.random.seed(RANDOM_STATE)


np.random.seed(RANDOM_STATE)  # global numpy seed for reproducibility


PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'code' else Path.cwd().resolve()

DATA_PATH = PROJECT_ROOT / 'data' / 'diabetic_data.csv'

IDS_PATH = PROJECT_ROOT / 'data' / 'IDS_mapping.csv'

OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'readmission_analysis_notebook'

FIGURE_DIR = OUTPUT_DIR / 'figures'

TABLE_DIR = OUTPUT_DIR / 'tables'

MODEL_DIR = OUTPUT_DIR / 'models'

BAYES_DIR = OUTPUT_DIR / 'bayesian'

REPORT_DIR = OUTPUT_DIR / 'report'

CACHE_DIR = OUTPUT_DIR / '.cache'



for p in [OUTPUT_DIR, FIGURE_DIR, TABLE_DIR, MODEL_DIR, BAYES_DIR, REPORT_DIR, CACHE_DIR]:

    p.mkdir(parents=True, exist_ok=True)



sns.set_theme(style='whitegrid')

print('Project root:', PROJECT_ROOT)

print('Data path exists:', DATA_PATH.exists())

In [ ]:
# Kaggle provides 4 CPU cores and 30 GB RAM.
# ACTIVE_THREADS is overridden to use all available cores.
# OUTER_FOLDS is reduced from 5 to 3 to fit within the 12-hour Kaggle
# session limit while keeping nested CV scientifically valid.
# Remove or comment out these overrides when running locally.

import os, multiprocessing

ACTIVE_THREADS = multiprocessing.cpu_count()
OUTER_FOLDS    = 3

os.environ['OMP_NUM_THREADS']      = str(ACTIVE_THREADS)
os.environ['MKL_NUM_THREADS']      = str(ACTIVE_THREADS)
os.environ['OPENBLAS_NUM_THREADS'] = str(ACTIVE_THREADS)

print(f'Active threads : {ACTIVE_THREADS}')
print(f'Outer folds    : {OUTER_FOLDS}')
print(f'Optuna trials  : 50')

## 1 : Data Loading and Quality Report

This step loads the diabetic readmission dataset and compiles the initial quality report. Huang et al. (2021), Le et al. (2025), and Hasan et al. (2010) motivate transparent cohort definition and readiness diagnostics before modeling. Outputs are the raw preview, missingness counts, column summaries, and saved quality report artifacts.

In [ ]:
raw_df = pd.read_csv(DATA_PATH, na_values=['?'], low_memory=False)
ids_mapping = load_ids_mapping(IDS_PATH)

print('Dataset shape:', raw_df.shape)
display(raw_df.head())
display(raw_df.dtypes.astype(str).rename('dtype').to_frame())
display(raw_df.describe(include='all').transpose().head(20))

missing_counts = raw_df.isna().sum().rename('missing_count').reset_index().rename(columns={'index': 'column'})
duplicate_count = int(raw_df.duplicated().sum())

categorical_distributions = {}
for col in raw_df.select_dtypes(include=['object']).columns:
    categorical_distributions[col] = raw_df[col].fillna('Missing').value_counts().head(15).astype(int).to_dict()

data_quality_report = {
    'n_rows': int(raw_df.shape[0]),
    'n_columns': int(raw_df.shape[1]),
    'duplicates': duplicate_count,
    'missing_by_column': missing_counts.to_dict(orient='records'),
    'categorical_distributions': categorical_distributions,
}
save_json(data_quality_report, REPORT_DIR / 'data_quality_report.json')
missing_counts.to_csv(TABLE_DIR / 'missing_counts.csv', index=False)
print('Duplicate rows:', duplicate_count)

## 2 : Target Variable Definition

This step defines the binary 30-day readmission endpoint used for supervised learning. Huang et al. (2021), Le et al. (2025), and Hasan et al. (2010) support explicit endpoint definition to preserve comparability across studies. Outputs are the transformed target field and saved class-distribution summaries.

In [ ]:
df = raw_df.drop_duplicates().reset_index(drop=True).copy()
df['readmitted_30d'] = (df['readmitted'] == '<30').astype(int)

original_outcome_distribution = df['readmitted'].value_counts().to_dict()
binary_outcome_distribution = df['readmitted_30d'].value_counts().sort_index().to_dict()

print('Original outcome distribution:', original_outcome_distribution)
print('Binary outcome distribution:', binary_outcome_distribution)

## 3 : Preprocessing Pipeline

This step prepares training-only preprocessing components to avoid leakage into evaluation data. Huang et al. (2021) and Le et al. (2025) emphasize transparent feature handling, and Hasan et al. (2010) motivates clinically coherent predictor preparation. Outputs are feature matrices, grouped outcome vectors, and reusable preprocessing objects.

In [ ]:
feature_columns = [c for c in df.columns if c not in {'readmitted', 'readmitted_30d'}]
X_raw = df[feature_columns].copy()
y = df['readmitted_30d'].copy()
groups = df['patient_nbr'].copy()

print('Feature matrix shape:', X_raw.shape)
print('Target prevalence:', float(y.mean()))

## 4 : Exploratory Data Analysis

This step profiles cleaned predictors through distributional and domain-focused summaries. Huang et al. (2021) highlights reporting heterogeneity, Le et al. (2025) supports broad pre-model diagnostics, and Hasan et al. (2010) motivates clinically interpretable variable review. Outputs are EDA figures, status-stratified summaries, and predictor-domain tables.

In [ ]:
# EDA plots
eda_bundle = EncodedDataBundle(ids_mapping=ids_mapping)
eda_bundle.fit(X_raw)
exploratory_frame = eda_bundle.transform_clean_dataframe(X_raw, include_ids=False)

numeric_columns = exploratory_frame.select_dtypes(include=['number']).columns.tolist()
summary_by_target = exploratory_frame[numeric_columns].assign(readmitted_30d=y).groupby('readmitted_30d').agg(['mean', 'std'])
summary_by_target.to_csv(TABLE_DIR / 'summary_by_readmission_status.csv')

plot_numeric_distributions(exploratory_frame, numeric_columns, y, FIGURE_DIR / 'numeric_distributions.png')
plot_correlation_heatmap(exploratory_frame, numeric_columns, FIGURE_DIR / 'numeric_correlation_heatmap.png')
plot_category_frequencies(
    exploratory_frame,
    [c for c in ['race', 'gender', 'admission_type_id', 'discharge_disposition_id', 'medical_specialty', 'diag_1'] if c in exploratory_frame.columns],
    FIGURE_DIR / 'categorical_frequencies.png',
)

print('EDA artifacts saved to:', FIGURE_DIR)

## 5 : Train and Test Split

This step creates grouped development and hold-out partitions for leakage-aware evaluation. Huang et al. (2021) warns that simplistic splitting inflates performance estimates, Le et al. (2025) motivates stronger validation design, and Hasan et al. (2010) provides benchmark context. Outputs are train-test partitions and saved split summaries.

In [ ]:
holdout_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_index, test_index = next(holdout_splitter.split(X_raw, y, groups))

X_train_raw = X_raw.iloc[train_index].reset_index(drop=True)
X_test_raw = X_raw.iloc[test_index].reset_index(drop=True)
y_train = y.iloc[train_index].reset_index(drop=True)
y_test = y.iloc[test_index].reset_index(drop=True)
groups_train = groups.iloc[train_index].reset_index(drop=True)

print('Train size:', len(X_train_raw), 'Test size:', len(X_test_raw))
print('Train positive rate:', float(y_train.mean()))
print('Test positive rate:', float(y_test.mean()))

## 6 : Class Imbalance Handling

This step applies SMOTE within training data to address outcome imbalance. Huang et al. (2021) notes imbalance handling is often underreported, Le et al. (2025) compares resampling and cost-sensitive strategies, and Hasan et al. (2010) provides the clinical baseline comparator. Outputs are balanced training matrices and saved class-distribution tables.

In [ ]:
bundle = EncodedDataBundle(ids_mapping=ids_mapping)
X_train_encoded = bundle.fit_transform(X_train_raw)
X_test_encoded = bundle.transform(X_test_raw)

cleaned_dataset = bundle.transform_clean_dataframe(X_raw, include_ids=False)
cleaned_dataset['readmitted_30d'] = y.to_numpy()
cleaned_dataset.to_csv(OUTPUT_DIR / 'cleaned_dataset.csv', index=False)

X_train_balanced_preview, y_train_balanced_preview = maybe_apply_smote(
    X_train_encoded.astype(np.float32),
    y_train.to_numpy(dtype=np.int32),
    RANDOM_STATE,
)

smote_report = pd.DataFrame([
    {'dataset': 'train_before_smote', 'class_0': int((y_train == 0).sum()), 'class_1': int((y_train == 1).sum())},
    {'dataset': 'train_after_smote', 'class_0': int((y_train_balanced_preview == 0).sum()), 'class_1': int((y_train_balanced_preview == 1).sum())},
])
smote_report.to_csv(TABLE_DIR / 'smote_class_distribution.csv', index=False)
display(smote_report)

## 7 : Feature Selection

This step selects predictors using a combined LASSO and Boruta strategy. Huang et al. (2021) and Le et al. (2025) support principled feature reduction, and Hasan et al. (2010) motivates parsimonious clinical predictors for interpretability. Outputs are feature-selection tables, selected-feature lists, and reduced modeling matrices.

In [ ]:
selector = CombinedFeatureSelector(random_state=RANDOM_STATE)

selector.fit(X_train_encoded, y_train.to_numpy(dtype=np.int32), bundle.feature_names_)

selector.selection_table_.to_csv(TABLE_DIR / 'feature_selection_summary.csv', index=False)
# Saves per-feature LASSO and Boruta selection flags  required by Le et al. (2025)


X_train_selected = selector.transform(X_train_encoded)

X_test_selected = selector.transform(X_test_encoded)

selected_feature_names = selector.selected_feature_names_



selector.selection_table_.to_csv(TABLE_DIR / 'feature_selection_table.csv', index=False)

pd.DataFrame({'selected_feature': selected_feature_names}).to_csv(TABLE_DIR / 'selected_features.csv', index=False)

print('Selected feature count:', len(selected_feature_names))

display(pd.DataFrame({'selected_feature': selected_feature_names}).head(20))

### 7c : Feature Selection Agreement

The agreement plot shows which features were selected by LASSO alone, Boruta alone, or both methods simultaneously, making the union selection strategy transparent and auditable. Le et al. (2025) use combined LASSO and tree-based selection to balance linear and non-linear feature relevance. The output is an agreement figure saved to FIGURE_DIR.

In [ ]:
# Feature selection agreement plot

sel_table = selector.selection_table_.copy()



sel_table['group'] = np.select(

    [

        sel_table['lasso_selected'] & sel_table['boruta_selected'],

        sel_table['lasso_selected'] & (~sel_table['boruta_selected']),

        (~sel_table['lasso_selected']) & sel_table['boruta_selected'],

    ],

    ['Both', 'LASSO only', 'Boruta only'],

    default='Not selected',

)



agreement_counts = (

    sel_table[sel_table['group'].isin(['Both', 'LASSO only', 'Boruta only'])]

    .groupby('group').size().reindex(['LASSO only', 'Boruta only', 'Both']).fillna(0)

)



fig, axes = plt.subplots(1, 2, figsize=(15, 6))



axes[0].bar(agreement_counts.index, agreement_counts.values, color=['#5dade2', '#58d68d', '#f5b041'])

axes[0].set_title('Selection Agreement Counts')

axes[0].set_ylabel('Feature count')



plot_top = sel_table.sort_values('combined_score', ascending=False).head(30)

sns.scatterplot(data=plot_top, x='combined_score', y='feature', hue='group', palette={'Both': '#f39c12', 'LASSO only': '#3498db', 'Boruta only': '#2ecc71', 'Not selected': '#95a5a6'}, ax=axes[1])

axes[1].set_title('Top 30 Features by Combined Score')

axes[1].set_xlabel('Combined score')

axes[1].set_ylabel('Feature')



plt.tight_layout()

plt.savefig(FIGURE_DIR / 'feature_selection_agreement.png', dpi=200, bbox_inches='tight')

plt.show()
plt.close()

## 8 : Model Development and Nested Cross-Validation

This step trains candidate machine-learning models with grouped nested cross-validation and Optuna tuning. Huang et al. (2021), Le et al. (2025), and Hasan et al. (2010) support broad benchmarking with leakage-aware validation. Outputs are tuned models, fold-level metrics, trial tables, and hold-out prediction frames.

In [ ]:
# Model development and nested CV
tuning_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + 11)
_, tuning_subset_index = next(tuning_splitter.split(X_train_raw, y_train, groups_train))
X_tuning_raw = X_train_raw.iloc[tuning_subset_index].reset_index(drop=True)
y_tuning = y_train.iloc[tuning_subset_index].reset_index(drop=True)
groups_tuning = groups_train.iloc[tuning_subset_index].reset_index(drop=True)

fold_datasets = prepare_fold_datasets(X_tuning_raw, y_tuning, groups_tuning, ids_mapping)

model_names = [
    'logistic_regression',
    'random_forest',
    'xgboost',
    'lightgbm',
    'catboost',
    'mlp',
]

nested_cv_frames = []
best_params_registry = {}
final_models = {}
prediction_frames = {}
performance_rows = []

X_final_fit, y_final_fit = maybe_apply_smote(X_train_selected, y_train.to_numpy(dtype=np.int32), RANDOM_STATE)

for model_name in model_names:
    nested_metrics, best_params_list, trials_tables = nested_cv_for_model(model_name, fold_datasets)
    nested_metrics['model'] = model_name
    nested_cv_frames.append(nested_metrics)

    pd.concat(trials_tables, ignore_index=True).to_csv(TABLE_DIR / f'{model_name}_optuna_trials.csv', index=False)
    aggregated_params = aggregate_best_params(best_params_list)
    best_params_registry[model_name] = aggregated_params

    model = build_model(model_name, aggregated_params, seed=RANDOM_STATE)
    model.fit(X_final_fit, y_final_fit)

    probabilities = model.predict_proba(X_test_selected)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)

    metrics = compute_metrics(y_test.to_numpy(dtype=np.int32), probabilities)
    metrics['model'] = model_name
    performance_rows.append(metrics)

    prediction_frames[model_name] = pd.DataFrame({
        'y_true': y_test.to_numpy(dtype=np.int32),
        'probability': probabilities,
        'prediction': predictions,
    })
    final_models[model_name] = model

nested_cv_results = pd.concat(nested_cv_frames, ignore_index=True)
nested_cv_results.to_csv(TABLE_DIR / 'nested_cv_results.csv', index=False)
nested_cv_summary = (
    nested_cv_results.groupby('model')[['f1', 'auroc', 'precision', 'recall', 'accuracy', 'pr_auc', 'brier']]
    .mean()
    .reset_index()
    .sort_values('f1', ascending=False)
)
nested_cv_summary.to_csv(TABLE_DIR / 'nested_cv_summary.csv', index=False)
display(nested_cv_summary)

## 9 : Model Evaluation

This step evaluates hold-out discrimination and thresholded classification behavior. Huang et al. (2021) identifies AUROC as a dominant reporting metric, Le et al. (2025) broadens evaluation with precision-recall analysis, and Hasan et al. (2010) provides a conventional readmission benchmark. Outputs are comparative metric tables and ROC, PR, and confusion-matrix figures.

In [ ]:
# Model evaluation with ROC, PR, and confusion matrices

evaluation_rows = []

for model_name, frame in prediction_frames.items():

    metrics = compute_metrics(frame['y_true'].to_numpy(dtype=np.int32), frame['probability'].to_numpy())

    metrics['model'] = model_name

    evaluation_rows.append(metrics)



performance_table = pd.DataFrame(evaluation_rows).sort_values('f1', ascending=False).reset_index(drop=True)

performance_table.to_csv(TABLE_DIR / 'model_performance_summary.csv', index=False)
# Saves primary evaluation metrics for all models  required for reproducibility
performance_table.to_csv(TABLE_DIR / 'model_performance_comparison.csv', index=False)



plot_roc_curves(prediction_frames, FIGURE_DIR / 'roc_curves.png')

plot_pr_curves(prediction_frames, FIGURE_DIR / 'precision_recall_curves.png')

plot_confusion_matrices(prediction_frames, FIGURE_DIR / 'confusion_matrices.png')



display(pd.DataFrame(performance_table))

### 9a : Nested CV Boxplot

Nested cross-validation AUROC distributions are shown as boxplots for each model, visualising variance in generalisation performance across outer folds. Huang et al. (2021) report that nested CV is the minimum standard for unbiased model comparison in medical prediction studies. The output is a boxplot figure saved to FIGURE_DIR.

In [ ]:
# Nested CV boxplot

sns.set_theme(style='whitegrid')



fig, axes = plt.subplots(1, 2, figsize=(16, 6))



sns.boxplot(data=nested_cv_results, x='model', y='auroc', ax=axes[0], color='#aed6f1')

sns.stripplot(data=nested_cv_results, x='model', y='auroc', ax=axes[0], color='#2c3e50', alpha=0.6, jitter=True)

axes[0].axhline(0.68, color='red', linestyle='--', linewidth=1.2)

axes[0].set_title('Nested CV AUROC Distribution per Model (Huang et al. 2021 median = 0.68)')

axes[0].tick_params(axis='x', rotation=45)



sns.boxplot(data=nested_cv_results, x='model', y='f1', ax=axes[1], color='#fad7a0')

sns.stripplot(data=nested_cv_results, x='model', y='f1', ax=axes[1], color='#2c3e50', alpha=0.6, jitter=True)

axes[1].set_title('Nested CV F1 Distribution per Model')

axes[1].tick_params(axis='x', rotation=45)



plt.tight_layout()

plt.savefig(FIGURE_DIR / 'nested_cv_boxplot.png', dpi=200, bbox_inches='tight')

plt.show()
plt.close()

### 9b : Extended Metrics Table

An extended metrics table reports PR-AUC, F2, balanced accuracy, and balanced Brier score alongside AUROC for every model, providing the multi-metric view recommended for imbalanced clinical datasets. Le et al. (2025) report all five of these metrics in their comparative evaluation. The output is a formatted table displayed inline and saved to TABLE_DIR.

In [ ]:
# Extended metrics table
# Required by Le et al. (2025): all four metrics used in their evaluation framework.
# F2 score weights recall twice as heavily as precision (better for imbalanced data).

from sklearn.metrics import average_precision_score, fbeta_score, balanced_accuracy_score

extended_rows = []
for model_name, frame in prediction_frames.items():
    y_true = frame['y_true'].to_numpy()
    y_prob = frame['probability'].to_numpy()
    y_pred = frame['prediction'].to_numpy()

    prauc = float(average_precision_score(y_true, y_prob))
    f2 = float(fbeta_score(y_true, y_pred, beta=2, zero_division=0))
    bal_acc = float(balanced_accuracy_score(y_true, y_pred))

    # Balanced Brier: stratified sum over both classes
    pos_idx = y_true == 1
    neg_idx = y_true == 0
    brier_pos = float(np.mean((y_prob[pos_idx] - 1) ** 2)) if pos_idx.sum() > 0 else 0.0
    brier_neg = float(np.mean((y_prob[neg_idx] - 0) ** 2)) if neg_idx.sum() > 0 else 0.0
    bal_brier = brier_pos + brier_neg

    extended_rows.append({
        'model': model_name,
        'prauc': prauc,
        'f2': f2,
        'balanced_accuracy': bal_acc,
        'balanced_brier': bal_brier,
    })

extended_metrics_frame = pd.DataFrame(extended_rows)
base_performance = performance_table.copy() if 'performance_table' in globals() else pd.DataFrame(performance_rows)
extended_performance = base_performance.merge(extended_metrics_frame, on='model', how='left')
extended_performance.to_csv(TABLE_DIR / 'extended_model_performance.csv', index=False)

display_columns = [c for c in ['model', 'auroc', 'prauc', 'f1', 'f2', 'precision', 'recall', 'balanced_accuracy', 'brier', 'balanced_brier'] if c in extended_performance.columns]
display(pd.DataFrame(extended_performance[display_columns]))

### 9c : Performance Comparison Chart

This section compares key model metrics in a grouped chart to support rapid cross-model interpretation.
Huang et al. (2021) recommends multi-metric comparison across model classes rather than relying on a single discrimination estimate.
Output is a model performance comparison figure saved to FIGURE_DIR.

In [ ]:
# Performance comparison bar chart

sns.set_theme(style='whitegrid')



metric_cols = [c for c in ['auroc', 'prauc', 'f1', 'f2', 'balanced_accuracy'] if c in extended_performance.columns]

plot_perf = extended_performance[['model'] + metric_cols].copy()

plot_long = plot_perf.melt(id_vars='model', value_vars=metric_cols, var_name='metric', value_name='value')



fig, ax = plt.subplots(figsize=(14, 6))

sns.barplot(data=plot_long, x='model', y='value', hue='metric', ax=ax)

ax.axhline(0.68, color='red', linestyle='--', linewidth=1.2, label='Huang 2021 median AUROC')

ax.axhline(0.737, color='purple', linestyle='--', linewidth=1.2, label='Le 2025 CatBoost AUROC')

ax.set_title('Model Performance Comparison Across Metrics')

ax.set_xlabel('Model')

ax.set_ylabel('Score')

ax.tick_params(axis='x', rotation=45)

ax.legend(loc='best')



plt.tight_layout()

plt.savefig(FIGURE_DIR / 'model_performance_comparison.png', dpi=200, bbox_inches='tight')

plt.show()
plt.close()

### 9d : DeLong Pairwise Significance

This section tests whether pairwise AUROC differences are statistically meaningful across competing models.
Le et al. (2025) uses DeLong testing with multiplicity control to distinguish robust ranking differences from sampling noise.
Outputs are a DeLong significance heatmap in FIGURE_DIR and pairwise test tables in TABLE_DIR.

In [ ]:
# Pairwise DeLong test for AUROC comparison
# Le et al. (2025) used DeLong test + Bonferroni correction for significance testing.

from itertools import combinations
from scipy import stats

def delong_roc_variance(ground_truth, predictions):
    """DeLong variance estimation for AUC (simplified Mann-Whitney form)."""
    n1 = int(ground_truth.sum())
    n0 = int((1 - ground_truth).sum())
    pos_scores = predictions[ground_truth == 1]
    neg_scores = predictions[ground_truth == 0]

    if n1 < 2 or n0 < 2:
        auc = float(np.nan)
        return auc, float(np.nan)

    kernel = np.array([
        np.mean((p > neg_scores).astype(float) + 0.5 * (p == neg_scores).astype(float))
        for p in pos_scores
    ])
    auc = kernel.mean()
    v10 = np.var(kernel, ddof=1) / n1

    kernel2 = np.array([
        np.mean((pos_scores > n).astype(float) + 0.5 * (pos_scores == n).astype(float))
        for n in neg_scores
    ])
    v01 = np.var(kernel2, ddof=1) / n0
    return float(auc), float(v10 + v01)

ml_models = [m for m in prediction_frames if m not in ('bayesian_logistic', 'bayesian_bart')]
pairs = list(combinations(ml_models, 2))
delong_rows = []

y_true_np = y_test.to_numpy(dtype=int)
for m1, m2 in pairs:
    p1 = prediction_frames[m1]['probability'].to_numpy()
    p2 = prediction_frames[m2]['probability'].to_numpy()
    auc1, var1 = delong_roc_variance(y_true_np, p1)
    auc2, var2 = delong_roc_variance(y_true_np, p2)

    if np.isnan(var1) or np.isnan(var2):
        z = np.nan
        p = np.nan
    else:
        z = (auc1 - auc2) / np.sqrt(var1 + var2 + 1e-12)
        p = float(2 * (1 - stats.norm.cdf(abs(z))))

    delong_rows.append(
        {
            'model_1': m1,
            'model_2': m2,
            'auroc_1': round(auc1, 4) if not np.isnan(auc1) else np.nan,
            'auroc_2': round(auc2, 4) if not np.isnan(auc2) else np.nan,
            'z_stat': round(float(z), 4) if not np.isnan(z) else np.nan,
            'p_value': round(float(p), 4) if not np.isnan(p) else np.nan,
        }
    )

delong_frame = pd.DataFrame(delong_rows)

n_comparisons = max(len(delong_rows), 1)
delong_frame['p_bonferroni'] = (delong_frame['p_value'] * n_comparisons).clip(upper=1.0).round(4)
delong_frame['significant'] = delong_frame['p_bonferroni'] < 0.05
delong_frame.to_csv(TABLE_DIR / 'delong_pairwise_auroc_tests.csv', index=False)

display(pd.DataFrame(delong_frame.sort_values('p_value', na_position='last')))

### 9e : DeLong Pairwise Heatmap

Pairwise DeLong significance results with Bonferroni correction are visualised as a heatmap, allowing rapid identification of which model pairs are statistically distinguishable. Le et al. (2025) present pairwise significance heatmaps as a standard component of multi-model comparative reporting. The output is a heatmap figure saved to FIGURE_DIR.

In [ ]:
# DeLong pairwise heatmap

sns.set_theme(style='whitegrid')



models = sorted(set(delong_frame['model_1']).union(set(delong_frame['model_2'])))

pmat = pd.DataFrame(np.nan, index=models, columns=models)



for _, row in delong_frame.iterrows():

    m1, m2, pv = row['model_1'], row['model_2'], row['p_bonferroni']

    pmat.loc[m1, m2] = pv

    pmat.loc[m2, m1] = pv



for m in models:

    pmat.loc[m, m] = np.nan



mask = np.eye(len(models), dtype=bool)

fig, ax = plt.subplots(figsize=(8, 7))

sns.heatmap(pmat, annot=True, fmt='.3f', cmap='RdYlGn_r', vmin=0, vmax=0.05, mask=mask, ax=ax)

ax.set_title('Pairwise DeLong Test p-values (Bonferroni corrected) Ã¢â‚¬â€ red = significant difference')



plt.tight_layout()

plt.savefig(FIGURE_DIR / 'delong_pairwise_heatmap.png', dpi=200, bbox_inches='tight')

plt.show()
plt.close()

## 10 : Calibration

This step assesses agreement between predicted probabilities and observed outcomes. Huang et al. (2021) reports calibration as underemphasized, Le et al. (2025) supports probability-level evaluation beyond discrimination, and Hasan et al. (2010) motivates interpretable risk estimates for clinical use. Outputs are calibration summaries, curve-point tables, and calibration figures.

In [ ]:
# Calibration summary and calibration curves
calibration_rows = []
calibration_curve_rows = []

for model_name, frame in prediction_frames.items():
    ece, curve = expected_calibration_error(frame['y_true'].to_numpy(), frame['probability'].to_numpy())
    curve['model'] = model_name
    calibration_curve_rows.append(curve.assign(calibration='uncalibrated'))
    calibration_rows.append({
        'model': model_name,
        'calibration': 'uncalibrated',
        'brier': brier_score_loss(frame['y_true'], frame['probability']),
        'ece': ece,
    })

top_models_for_calibration = [model for model in performance_table['model'].tolist() if model in best_params_registry][:CALIBRATION_TOP_MODELS]
for model_name in top_models_for_calibration:
    calibration_models = fit_calibration_models(
        model_name=model_name,
        params=best_params_registry[model_name],
        X_train=X_train_selected,
        y_train=y_train.to_numpy(dtype=np.int32),
        groups_train=groups_train.to_numpy(),
    )
    raw_probabilities = prediction_frames[model_name]['probability'].to_numpy()
    calibrated_probs = {
        'platt': calibration_models['platt'].predict_proba(raw_probabilities.reshape(-1, 1))[:, 1],
        'isotonic': calibration_models['isotonic'].predict(raw_probabilities),
    }

    for method, probs in calibrated_probs.items():
        ece, curve = expected_calibration_error(y_test.to_numpy(dtype=np.int32), probs)
        curve['model'] = f'{model_name}_{method}'
        calibration_curve_rows.append(curve.assign(calibration=method))
        calibration_rows.append({
            'model': model_name,
            'calibration': method,
            'brier': brier_score_loss(y_test, probs),
            'ece': ece,
        })

calibration_summary = pd.DataFrame(calibration_rows).sort_values(['model', 'brier'])
calibration_summary.to_csv(TABLE_DIR / 'calibration_summary.csv', index=False)

calibration_curve_frame = pd.concat(calibration_curve_rows, ignore_index=True)
calibration_curve_frame.to_csv(TABLE_DIR / 'calibration_curve_points.csv', index=False)

plot_calibration_curves(calibration_curve_frame, FIGURE_DIR / 'calibration_curves.png')

display(pd.DataFrame(calibration_summary.head(20)))

### 10d : Cumulative Gain Chart

The cumulative gain chart shows the proportion of true readmission cases captured as a function of the proportion of patients screened, quantifying operational value for targeted intervention programmes. Huang et al. (2021) identify gain charts as the primary tool for translating model performance into clinical resource allocation decisions. The output is a gain chart figure saved to FIGURE_DIR.

In [ ]:
# Cumulative gain chart (top 3 models by AUROC)

sns.set_theme(style='whitegrid')



top3_auc_models = nested_cv_summary.sort_values('auroc', ascending=False).head(3)['model'].tolist()



fig, ax = plt.subplots(figsize=(10, 7))

for model_name in top3_auc_models:

    frame = prediction_frames[model_name].copy().sort_values('probability', ascending=False).reset_index(drop=True)

    y_true = frame['y_true'].to_numpy()

    cum_pos = np.cumsum(y_true)

    total_pos = max(cum_pos[-1], 1)

    frac_pop = (np.arange(1, len(frame) + 1)) / len(frame)

    cum_recall = cum_pos / total_pos

    ax.plot(frac_pop, cum_recall, label=model_name)



ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random baseline')

ax.plot([0, total_pos / len(frame), 1], [0, 1, 1], linestyle=':', color='black', label='Perfect model')

ax.set_title('Cumulative Gain Chart (Top 3 Models)')

ax.set_xlabel('Fraction of population screened')

ax.set_ylabel('Fraction of positives captured')

ax.legend(loc='lower right')



best_model_for_gain = top3_auc_models[0]

best_frame = prediction_frames[best_model_for_gain].copy().sort_values('probability', ascending=False).reset_index(drop=True)

y_best = best_frame['y_true'].to_numpy()

cut = max(int(0.2 * len(best_frame)), 1)

cap = y_best[:cut].sum() / max(y_best.sum(), 1)

ax.text(0.02, 0.05, f'Top 20% captures {cap * 100:.1f}% of readmissions ({best_model_for_gain})', transform=ax.transAxes, fontsize=10, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))



plt.tight_layout()

plt.savefig(FIGURE_DIR / 'cumulative_gain_chart.png', dpi=200, bbox_inches='tight')

plt.show()
plt.close()

## 11 : Explainability

This step explains the best tree-based model with SHAP attribution analysis. Le et al. (2025) uses SHAP as a core interpretability framework, Huang et al. (2021) highlights explainability barriers to adoption, and Hasan et al. (2010) motivates interpretable clinical modeling. Outputs are SHAP importance tables and summary, dependence, and local explanation figures.

In [ ]:
tree_candidates = [m for m in ['xgboost', 'lightgbm', 'catboost', 'random_forest'] if m in performance_table['model'].tolist()]
best_tree_model_name = performance_table[performance_table['model'].isin(tree_candidates)].iloc[0]['model']

shap_importance = shap_analysis(
    model_name=best_tree_model_name,
    model=final_models[best_tree_model_name],
    X_reference=X_test_selected,
    feature_names=selected_feature_names,
    figure_dir=FIGURE_DIR,
    table_dir=TABLE_DIR,
)
display(shap_importance.head(20))

### 11b : SHAP Waterfall

A SHAP waterfall plot for the highest-risk patient in the test set decomposes the individual prediction into additive feature contributions, providing a patient-level explainability view. Le et al. (2025) use waterfall plots to demonstrate that feature contributions are clinically interpretable at the individual level. The output is a waterfall figure saved to FIGURE_DIR.

In [ ]:
# SHAP waterfall for representative high-risk patient

import shap



best_tree_candidates = [m for m in ['catboost', 'xgboost', 'lightgbm', 'random_forest'] if m in final_models]

if not best_tree_candidates:

    print('No tree-based model available for SHAP waterfall.')

else:

    best_tree = performance_table[performance_table['model'].isin(best_tree_candidates)].iloc[0]['model']

    probs = prediction_frames[best_tree]['probability'].to_numpy()

    target_prob = np.quantile(probs, 0.9)

    representative_idx = int(np.argmin(np.abs(probs - target_prob)))



    model = final_models[best_tree]

    explainer = shap.Explainer(model, X_train_selected)

    shap_values = explainer(X_test_selected)



    plt.figure(figsize=(10, 6))

    shap.plots.waterfall(shap_values[representative_idx], max_display=15, show=False)

    plt.title('SHAP Waterfall, Representative High-Risk Patient (90th percentile predicted risk)')

    plt.tight_layout()

    plt.savefig(FIGURE_DIR / 'shap_waterfall_high_risk_patient.png', dpi=200, bbox_inches='tight')

    plt.show()
plt.close()

### 13d : LACE Proxy Benchmark

The LACE proxy score is computed from available dataset variables and evaluated as a baseline comparator, contextualising pipeline performance against this widely used clinical readmission rule. Hasan et al. (2010) establish LACE as the standard clinical benchmark for general medicine readmission prediction. The output is a benchmark AUROC comparison displayed inline.

In [ ]:
# LACE proxy benchmark
from sklearn.metrics import roc_auc_score, average_precision_score

def compute_lace_proxy(df_raw, y):
    """
    Approximate LACE score from UCI diabetic dataset columns.
    L: time_in_hospital (capped 1-7)
    A: admission_type_id == 1 (emergency, +3)
    C: number_diagnoses (capped 0-5)
    E: number_emergency (capped 0-4)
    """
    lace = pd.Series(0.0, index=df_raw.index)
    if 'time_in_hospital' in df_raw.columns:
        lace += df_raw['time_in_hospital'].clip(1, 7)
    if 'admission_type_id' in df_raw.columns:
        lace += (df_raw['admission_type_id'] == 1).astype(float) * 3
    if 'number_diagnoses' in df_raw.columns:
        lace += df_raw['number_diagnoses'].clip(0, 5)
    if 'number_emergency' in df_raw.columns:
        lace += df_raw['number_emergency'].clip(0, 4)
    return lace

lace_scores = compute_lace_proxy(X_test_raw, y_test)
lace_auc = float(roc_auc_score(y_test.to_numpy(dtype=int), lace_scores))
lace_prauc = float(average_precision_score(y_test.to_numpy(dtype=int), lace_scores))

lace_benchmark = pd.DataFrame([
    {
        'model': 'LACE_proxy',
        'auroc': lace_auc,
        'prauc': lace_prauc,
        'note': 'Conventional LACE-type score proxy',
    }
])
lace_benchmark.to_csv(TABLE_DIR / 'lace_benchmark.csv', index=False)

ml_auroc_ref = float(performance_table['auroc'].max()) if 'performance_table' in globals() and not performance_table.empty else np.nan
lace_compare = pd.DataFrame([
    {'metric': 'lace_proxy_auroc', 'value': lace_auc},
    {'metric': 'lace_proxy_prauc', 'value': lace_prauc},
    {'metric': 'best_model_auroc', 'value': ml_auroc_ref},
    {'metric': 'auroc_difference', 'value': ml_auroc_ref - lace_auc if pd.notna(ml_auroc_ref) else np.nan},
])
display(pd.DataFrame(lace_benchmark))
display(pd.DataFrame(lace_compare))

## IEEE Paper Figures and Tables

The following outputs from this notebook map directly to the IEEE conference paper sections.

**Table 1  Model Performance:** `model_performance_summary.csv` in TABLE_DIR.
Reports AUROC, PR-AUC, F2, and Brier score for all models including LACE proxy.

**Figure 1  Pipeline Architecture:** Draw manually in draw.io using the pipeline
structure described in the methodology section.

**Figure 2  ROC Curves:** `roc_curves.png` in FIGURE_DIR.
Use top 3 models by AUROC plus LACE baseline.

**Figure 3  SHAP Feature Importance:** `shap_importance.png` in FIGURE_DIR.
Top 15 features shown.

**Table 2  DeLong Pairwise Significance:** `delong_pairwise.csv` in TABLE_DIR.

**Supplementary  Feature Selection:** `feature_selection_agreement.png` in FIGURE_DIR.

**Supplementary  Gain Chart:** `cumulative_gain_chart.png` in FIGURE_DIR.

## 16 : Save Outputs

This step persists trained objects, figures, and tables to structured output directories. Huang et al. (2021), Le et al. (2025), and Hasan et al. (2010) support reproducible artifact reporting for benchmark transparency. Outputs are saved model files, table exports, and report-ready visual artifacts.

In [ ]:
import joblib

for model_name, model in final_models.items():
    joblib.dump(model, MODEL_DIR / f'{model_name}_model.joblib')

joblib.dump(bundle, MODEL_DIR / 'preprocessing_bundle.joblib')
joblib.dump(selector, MODEL_DIR / 'feature_selector.joblib')

print('Saved all models and preprocessing artifacts to:', MODEL_DIR)

## 17 : Results and Discussion
Benchmark alignment is summarized against Huang et al. (2021), Le et al. (2025), and Hasan et al. (2010), covering discrimination, calibration, and transparent clinical comparison.

Performance benchmark table.
| Model | AUROC | Source |
|---|---|---|
| LACE proxy | Computed below | Notebook |
| Hasan logistic | 0.61-0.65 (Hasan et al., 2010) | Hasan et al. (2010) |
| Huang median | 0.68 (Huang et al., 2021) | Huang et al. (2021) |
| Le CatBoost | 0.737 (Le et al., 2025) | Le et al. (2025) |
| Le SuperLearner | 0.742 (Le et al., 2025) | Le et al. (2025) |
| Pipeline best | Computed below | Notebook |

Methodological checklist: diverse models, imbalance handling, calibration analysis, explainability outputs, and baseline benchmarking.

Limitations: LACE is proxy-derived and validation is temporal-internal rather than true external-site testing.


In [ ]:
# Master summary table (all models and contexts)
from sklearn.metrics import average_precision_score, fbeta_score, balanced_accuracy_score

required_order = [
    'logistic_regression',
    'random_forest',
    'xgboost',
    'lightgbm',
    'catboost',
    'mlp',
    'bayesian_logistic',
    'bayesian_bart',
    'super_learner',
    'cost_sensitive_primary',
    'lace_proxy',
]

master_rows = []
for model_name, frame in prediction_frames.items():
    y_true = frame['y_true'].to_numpy(dtype=np.int32)
    y_prob = frame['probability'].to_numpy()
    y_pred = frame['prediction'].to_numpy(dtype=np.int32)
    base = compute_metrics(y_true, y_prob)
    pos_idx = y_true == 1
    neg_idx = y_true == 0
    brier_pos = float(np.mean((y_prob[pos_idx] - 1) ** 2)) if pos_idx.sum() > 0 else 0.0
    brier_neg = float(np.mean((y_prob[neg_idx] - 0) ** 2)) if neg_idx.sum() > 0 else 0.0
    ece_value, _ = expected_calibration_error(y_true, y_prob)
    master_rows.append({
        'model': model_name,
        'auroc': float(base.get('auroc', np.nan)),
        'prauc': float(average_precision_score(y_true, y_prob)),
        'f1': float(base.get('f1', np.nan)),
        'f2': float(fbeta_score(y_true, y_pred, beta=2, zero_division=0)),
        'precision': float(base.get('precision', np.nan)),
        'recall': float(base.get('recall', np.nan)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'brier': float(base.get('brier', np.nan)),
        'balanced_brier': float(brier_pos + brier_neg),
        'ece': float(ece_value),
    })

if 'lace_benchmark' in globals() and not lace_benchmark.empty:
    lace_row = lace_benchmark.iloc[0]
    master_rows.append({
        'model': 'lace_proxy',
        'auroc': lace_row.get('auroc', np.nan),
        'prauc': lace_row.get('prauc', np.nan),
        'f1': np.nan,
        'f2': np.nan,
        'precision': np.nan,
        'recall': np.nan,
        'balanced_accuracy': np.nan,
        'brier': np.nan,
        'balanced_brier': np.nan,
        'ece': np.nan,
    })

master_table = pd.DataFrame(master_rows)
if not master_table.empty:
    master_table['model'] = master_table['model'].replace({'LACE_proxy': 'lace_proxy', 'statsmodels_logit': 'logistic_regression'})

for key in required_order:
    if master_table.empty or key not in master_table['model'].tolist():
        master_table = pd.concat([master_table, pd.DataFrame([{'model': key, 'auroc': np.nan, 'prauc': np.nan, 'f1': np.nan, 'f2': np.nan, 'precision': np.nan, 'recall': np.nan, 'balanced_accuracy': np.nan, 'brier': np.nan, 'balanced_brier': np.nan, 'ece': np.nan}])], ignore_index=True)

master_table = master_table.drop_duplicates(subset=['model'], keep='first').set_index('model').reindex(required_order).reset_index()
master_table = master_table[['model', 'auroc', 'prauc', 'f1', 'f2', 'precision', 'recall', 'balanced_accuracy', 'brier', 'balanced_brier', 'ece']]
master_table.to_csv(TABLE_DIR / 'master_model_comparison_table.csv', index=False)
display(pd.DataFrame(master_table))